# Dynamic Pricing of Diamonds
### Modeling how diamond attributes drive price, and simulating a quality-based dynamic pricing strategy


**Approach:** starting from a standard diamonds dataset, this notebook engineers a *dynamic pricing* feature (a quality premium applied to top-grade stones), then compares a classical econometric model (OLS regression) against a neural network to explain and predict price. It closes with a correlation analysis, a simulated pricing time series, and a breakdown of price distribution by cut.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


**Key libraries:** pandas (data handling), statsmodels (econometric modeling), matplotlib/seaborn (visualization), and scikit-learn (scaling, neural networks, and evaluation metrics).


### Data Input


In [ ]:
# Loading the provided dataset
df = pd.read_csv('Diamonds_Prices2022.csv').drop(columns=['Unnamed: 0'])


### Data Cleaning & Logical Conditions


In [ ]:
df = df[(df['x'] > 0) & (df['y'] > 0) & (df['z'] > 0)].copy()


In [ ]:
# Logical condition: flag high-quality diamonds for the dynamic pricing analysis
df['is_high_quality'] = ((df['cut'] == 'Ideal') &
                         (df['clarity'].isin(['IF', 'VVS1', 'VVS2']))).astype(int)


**Volume** is calculated as $x \times y \times z$. **Dynamic price** is the core transformation: it applies a 5% price premium specifically to diamonds flagged as high quality. **Log scaling** converts the dynamic price to a log scale (`log_price`) to stabilize variance for econometric modeling.


### Statistical Analysis


In [ ]:
# Volume calculation
df['volume'] = df['x'] * df['y'] * df['z']

# Dynamic pricing transformation: applying a quality premium
df['dynamic_price'] = df['price'] * (1 + 0.05 * df['is_high_quality'])

# Logarithmic scaling for econometric stability
df['log_price'] = np.log(df['dynamic_price'])


In [ ]:
# Mean, std dev, and coefficient of variation (CV)
stats = df[['dynamic_price', 'carat', 'volume']].agg(['mean', 'std'])
stats.loc['CV (%)'] = (stats.loc['std'] / stats.loc['mean']) * 100
print("--- Statistical Analysis ---\n", stats)


**Observation:** the data shows high volatility in price (CV of 101.38%), indicating a wide spread between budget and luxury diamonds.


### Regression Model


In [ ]:
# A. Simple linear regression
y = df['dynamic_price']
X_simple = sm.add_constant(df['carat'])
simple_res = sm.OLS(y, X_simple).fit()

# B. Multiple regression (checking significance of additional features)
X_multi = sm.add_constant(df[['carat', 'depth', 'table', 'volume', 'is_high_quality']])
multi_res = sm.OLS(y, X_multi.astype(float)).fit()
print("\n--- Multiple Regression Summary ---\n", multi_res.summary())


**R-squared (0.857):** the model explains approximately 85.7% of the variance in diamond prices. **Coefficients:** each carat increase adds ~\$7,746 to the price; the `is_high_quality` flag alone adds ~\$1,123 in value. **Multicollinearity note:** the high condition number ($1.03 \times 10^4$) suggests that features like carat and volume are highly correlated with each other.


### Neural Network


In [ ]:
# Preparing features and scaling (standardizing)
X_ml = df[['carat', 'depth', 'table', 'volume', 'is_high_quality']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_ml)


In [ ]:
# Split for model verification
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


**Standardization:** uses `StandardScaler` to normalize features (carat, depth, table, volume, quality flag) so the neural network can process them efficiently. **Split:** partitions the data into training (80%) and testing (20%) sets to validate the model's predictive power.


In [ ]:
# Train a small feedforward neural network and generate predictions on the held-out test set
nn_model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)
nn_model.fit(X_train, y_train)
y_pred_nn = nn_model.predict(X_test)

nn_r2 = r2_score(y_test, y_pred_nn)
nn_mse = mean_squared_error(y_test, y_pred_nn)
print(f"Neural Network R\u00b2: {nn_r2:.4f}")
print(f"Neural Network MSE: {nn_mse:,.2f}")


In [ ]:
# Actual vs Predicted (Neural Network Quality)
plt.figure(figsize=(9, 6))
plt.scatter(y_test, y_pred_nn, alpha=0.3, color='teal')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.title('Neural Network Performance: Actual vs Predicted Dynamic Price')
plt.xlabel('Actual Dynamic Price (USD)')
plt.ylabel('Predicted Dynamic Price (USD)')
plt.tight_layout()
plt.show()


### Correlation Analysis


In [ ]:
# Calculate correlation (numeric_only=True prevents the string-to-float error)
corr_matrix = df.corr(numeric_only=True)

# Set up the visual style
plt.figure(figsize=(12, 10))
sns.set_theme(style="white")

# Mask to hide the upper triangle (makes it easier to read)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Draw the heatmap
heatmap = sns.heatmap(corr_matrix,
                      mask=mask,
                      annot=True,
                      fmt=".2f",
                      cmap='coolwarm',
                      center=0,
                      linewidths=.5,
                      cbar_kws={"shrink": .8})

plt.title('Correlation Analysis of Diamond Pricing Features', fontsize=16)
plt.show()

# Print the strongest relationships with price
print("--- Top Correlations with Dynamic Price ---")
print(corr_matrix['dynamic_price'].sort_values(ascending=False))


**Observation:** carat and volume show a near-perfect positive correlation, confirming why the regression model flagged multicollinearity.


### Timeline Simulation


*Note: the dataset has no real transaction dates. The section below assigns synthetic, evenly-spaced timestamps to the existing rows purely to demonstrate time-series analysis technique — the resulting trend reflects row order, not actual historical pricing.*


In [ ]:
# Simulating a timeline to demonstrate time-series analysis on the dynamic pricing data
df['Date'] = pd.date_range(start='2025-01-01', periods=len(df), freq='h')
daily_trend = df.set_index('Date')['dynamic_price'].resample('D').mean()


This treats the static diamond data as if it were a log of transactions occurring sequentially, which makes it possible to demonstrate daily trend and volatility analysis — the same technique used on real transaction-level pricing data.


In [ ]:
plt.figure(figsize=(10, 5))
daily_trend.plot(color='darkblue')
plt.title('Simulated Daily Average Dynamic Price')
plt.xlabel('Date')
plt.ylabel('Average Dynamic Price (USD)')
plt.tight_layout()
plt.show()


**Volatility:** by calculating the daily trend, we can see whether the simulated 5% premium for high-quality diamonds (`is_high_quality`) produces a stable price trend, or whether fluctuations in the availability of "Ideal" cut diamonds create price spikes on certain simulated days — useful for demonstrating how a dynamic pricing strategy would show up in daily average price if applied to a real, time-stamped sales feed.


### Price Distribution by Cut


In [ ]:
# Setting the visual style
sns.set_theme(style="whitegrid")

# Create the boxplot
plt.figure(figsize=(12, 7))
sns.boxplot(x='cut', y='dynamic_price', hue='cut', data=df,
            order=['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'],
            palette='viridis', legend=False)

plt.title('Analysis of Dynamic Price Distribution by Diamond Cut', fontsize=15)
plt.xlabel('Diamond Cut (Quality)', fontsize=12)
plt.ylabel('Adjusted Dynamic Price (USD)', fontsize=12)

plt.show()


### Conclusion


- The OLS regression explains ~85.7% of price variance ($R^2 = 0.857$), with carat as the dominant driver (~\$7,746 per carat) and the high-quality flag adding a smaller, independent ~\$1,123.
- The neural network provides a non-linear comparison point against the linear OLS model on the same held-out test set (see the R²/MSE printed above).
- Carat and volume are near-perfectly correlated, which explains the multicollinearity flagged in the regression summary — a production version of this model would likely drop one of the two.
- The simulated 5% quality premium and the simulated timeline are both modeling techniques applied to a static dataset, standing in for the kind of live, timestamped data a real dynamic-pricing system would use.


In [ ]:
# Export the final processed dataframe to a CSV file
df.to_csv('Final_Dynamic_Pricing_Results.csv', index=False)
